In [ ]:
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    !pip install -q kaggle mlflow xgboost wandb

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

In [14]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [15]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

((421570, 5), (8190, 12), (45, 3))

In [16]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [17]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
]

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        return df[FEATURE_COLS]

In [18]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('XGBoost_Training')

<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/1', creation_time=1783246277587, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783246277587, lifecycle_stage='active', name='XGBoost_Training', tags={}, trace_location=None, workspace='default'>

In [19]:
with mlflow.start_run(run_name='XGBoost_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

missing_markdown_pct,▁
negative_sales_rows,▁
missing_markdown_pct,0.55849
negative_sales_rows,1285


In [20]:
with mlflow.start_run(run_name='XGBoost_Feature_Selection'):
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_Feature_Selection', reinit='finish_previous')

    builder = FeatureBuilder(stores, features)
    X_all = builder.transform(train)

    probe = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, n_jobs=-1)
    probe.fit(X_all, y_train)

    importances = pd.Series(probe.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
    mlflow.log_dict(importances.to_dict(), 'feature_importances.json')
    wandb.log({'feature_importances': wandb.Table(dataframe=importances.reset_index().rename(columns={'index': 'feature', 0: 'importance'}))})

    selected_features = importances[importances > 0.001].index.tolist()
    mlflow.log_param('n_features_selected', len(selected_features))
    mlflow.log_dict({'selected_features': selected_features}, 'selected_features.json')
    wandb.log({'n_features_selected': len(selected_features)})
    wandb.finish()

importances

n_features_selected,▁
n_features_selected,18


,0
Dept,0.254536
Size,0.215334
Type,0.082413
Store,0.077816
MarkDown3,0.069488
IsPreChristmas,0.058529
Month,0.047766
Week,0.047416
CPI,0.042407
IsHoliday,0.034607


In [21]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

[(np.datetime64('2010-10-01T00:00:00.000000000'),
  np.datetime64('2011-06-03T00:00:00.000000000')),
 (np.datetime64('2011-06-03T00:00:00.000000000'),
  np.datetime64('2012-02-03T00:00:00.000000000')),
 (np.datetime64('2012-02-03T00:00:00.000000000'),
  np.datetime64('2012-10-05T00:00:00.000000000'))]

In [22]:
params = dict(n_estimators=500, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_jobs=-1)

with mlflow.start_run(run_name='XGBoost_CV'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_CV', config=params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        train_mask = train['Date'] <= train_end
        val_mask = (train['Date'] > train_end) & (train['Date'] <= val_end)

        X_tr = builder.transform(train[train_mask])
        X_val = builder.transform(train[val_mask])
        y_tr = y_train[train_mask]
        y_val = y_train[val_mask]

        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        score = wmae(y_val.values, preds, train.loc[val_mask, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores))})
    wandb.finish()

fold_scores

fold,▁▅█
wmae,█▃▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,2555.84729
wmae_mean,3207.59955
wmae_std,662.59676


[np.float64(4116.541655675326),
 np.float64(2950.409689964599),
 np.float64(2555.8472924691096)]

fold,▁▅█
wmae,█▃▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,2555.84729
wmae_mean,3207.59955
wmae_std,662.59676


[np.float64(4116.541655675326),
 np.float64(2950.409689964599),
 np.float64(2555.8472924691096)]

In [23]:
with mlflow.start_run(run_name='XGBoost_Final'):
    mlflow.log_params(params)
    wandb.init(project='walmart-sales-forecasting', name='XGBoost_Final', config=params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilder(stores, features)),
        ('model', XGBRegressor(**params)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/05 10:13:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_wmae,▁
train_wmae,2139.95046


np.float64(2139.9504594068844)

2026/07/05 10:15:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_wmae,▁
train_wmae,2139.95046


np.float64(2139.9504594068844)